<a href="https://colab.research.google.com/github/katharinabopst-jpg/azure-ai-integrated-chatbot/blob/main/Azure_AI_Integrated_Chatbot_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install azure-ai-textanalytics==5.2.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.8/69.8 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 239.3/239.3 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.9/220.9 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.4/85.4 kB 5.8 MB/s eta 0:00:00


In [2]:
from azure.ai.textanalytics import TextAnalyticsClient
from azure.core.credentials import AzureKeyCredential
from getpass import getpass

AZURE_LANGUAGE_KEY = getpass("Enter your Azure Language key: ")
AZURE_LANGUAGE_ENDPOINT = input("Enter your Azure Language endpoint: ").strip()

client = TextAnalyticsClient(
    endpoint=AZURE_LANGUAGE_ENDPOINT,
    credential=AzureKeyCredential(AZURE_LANGUAGE_KEY)
)

print("Azure AI Language client created successfully.")

Enter your Azure Language key: ··········
Enter your Azure Language endpoint: https://kat-msai631-language.cognitiveservices.azure.com/
Azure AI Language client created successfully.


In [3]:
def analyze_sentiment(text):
    try:
        result = client.analyze_sentiment(
            documents=[text],
            show_opinion_mining=False
        )[0]

        if result.is_error:
            return {
                "sentiment": "unknown",
                "positive": 0,
                "neutral": 0,
                "negative": 0
            }

        return {
            "sentiment": result.sentiment,
            "positive": result.confidence_scores.positive,
            "neutral": result.confidence_scores.neutral,
            "negative": result.confidence_scores.negative
        }

    except Exception as e:
        return {
            "sentiment": "unavailable",
            "error": str(e)
        }


# Test the Azure AI Language connection
test_text = "I am really happy and excited about completing this project!"
result = analyze_sentiment(test_text)

print("Text:", test_text)
print("Azure AI Sentiment Result:", result)

Text: I am really happy and excited about completing this project!
Azure AI Sentiment Result: {'sentiment': 'positive', 'positive': 1.0, 'neutral': 0.0, 'negative': 0.0}


## Integrated Chatbot

The following implementation combines the original rule-based student support chatbot with Azure AI Language sentiment analysis. Topic-specific responses remain deterministic, while Azure provides sentiment classification and confidence scores that allow the chatbot to adapt its response tone.

# Azure AI-Integrated Student Support Chatbot

**MSAI 631: Artificial Intelligence for Human-Computer Interaction**  
**University of the Cumberlands**

## Project Overview

This project extends a traditional rule-based student support chatbot by integrating Microsoft Azure AI Language. The chatbot uses deterministic Python rules for topic recognition and response selection while Azure AI Language performs sentiment analysis on user input.

The resulting hybrid architecture demonstrates how traditional chatbot logic can be enhanced with a cloud-based artificial intelligence service. Azure AI sentiment analysis allows the system to identify positive, neutral, negative, or mixed sentiment and adapt the chatbot's response accordingly.

## Architecture

**User Input → Azure AI Language Sentiment Analysis → Rule-Based Topic Detection → Sentiment-Aware Response**

Azure credentials are entered securely at runtime and are not stored in the notebook or source code.

In [4]:
def sentiment_prefix(sentiment):
    if sentiment == "negative":
        return "I understand that this may be frustrating. "
    elif sentiment == "positive":
        return "It sounds like you are feeling positive about this. "
    elif sentiment == "mixed":
        return "It sounds like you have mixed feelings about this. "
    else:
        return ""


def chatbot_response(user_input, sentiment):
    user_input = user_input.lower().strip()
    prefix = sentiment_prefix(sentiment)

    if "assignment" in user_input:
        response = (
            "Assignments should be reviewed carefully for instructions, "
            "deadlines, and required submission materials."
        )

    elif "deadline" in user_input or "due" in user_input:
        response = (
            "Please check your course learning management system "
            "for the official assignment deadline."
        )

    elif "office hours" in user_input:
        response = (
            "Office hours vary by instructor. Please review your "
            "syllabus or course announcements."
        )

    elif "contact" in user_input or "email" in user_input:
        response = (
            "You can find instructor contact information in the "
            "course syllabus or learning management system."
        )

    elif "study" in user_input or "tips" in user_input:
        response = (
            "A good study strategy is to review material in small "
            "sections, take notes, and test yourself regularly."
        )

    elif any(word in user_input for word in ["hello", "hi", "hey"]):
        response = (
            "Hello! I am the Azure AI-Integrated Student Support Chatbot. "
            "Type 'help' to see my capabilities."
        )

    elif "help" in user_input or "capabilities" in user_input:
        response = (
            "I can help with assignments, deadlines, office hours, "
            "contact information, and study tips."
        )

    else:
        response = (
            "I am sorry, I did not understand that request. "
            "Type 'help' to see the topics I can answer."
        )

    return prefix + response


print("Azure AI-Integrated Student Support Chatbot")
print("Type 'help' to see available topics.")
print("Type 'bye', 'exit', or 'quit' to stop the chatbot.")

while True:
    user_message = input("\nYou: ").strip()

    if user_message == "":
        print("Bot: I did not receive a question. Please type 'help' to see what I can do.")
        continue

    if user_message.lower() in ["bye", "exit", "quit"]:
        print("Bot: Goodbye! Thank you for using the Student Support Chatbot.")
        break

    sentiment_data = analyze_sentiment(user_message)

    print(
        f"Azure AI Sentiment: {sentiment_data['sentiment']} "
        f"(positive={sentiment_data.get('positive', 0):.2f}, "
        f"neutral={sentiment_data.get('neutral', 0):.2f}, "
        f"negative={sentiment_data.get('negative', 0):.2f})"
    )

    response = chatbot_response(
        user_message,
        sentiment_data["sentiment"]
    )

    print("Bot:", response)

Azure AI-Integrated Student Support Chatbot
Type 'help' to see available topics.
Type 'bye', 'exit', or 'quit' to stop the chatbot.

You: I am frustrated and disappointed about the deadline.
Azure AI Sentiment: negative (positive=0.00, neutral=0.00, negative=1.00)
Bot: I understand that this may be frustrating. Please check your course learning management system for the official assignment deadline.

You: I am really happy that I finished my assignment!
Azure AI Sentiment: positive (positive=1.00, neutral=0.00, negative=0.00)
Bot: It sounds like you are feeling positive about this. Assignments should be reviewed carefully for instructions, deadlines, and required submission materials.

You: The office hours are listed in the syllabus.
Azure AI Sentiment: neutral (positive=0.00, neutral=1.00, negative=0.00)
Bot: Office hours vary by instructor. Please review your syllabus or course announcements.

You: I am excited about these study tips!
Azure AI Sentiment: positive (positive=1.00, neu

## Testing

The chatbot was tested using positive, negative, and neutral statements, unsupported input, empty input, and an exit command. Testing confirmed that Azure AI Language successfully returned sentiment classifications and confidence scores while the traditional rule-based component continued to control topic recognition and response selection.

## Security

The Azure AI Language API key is not hard-coded or stored in this notebook. The key is entered at runtime using Python's `getpass()` function to reduce the risk of exposing credentials when the source code is shared through GitHub.